# Лабораторная работа 8. Скрапинг и анализ текста

Выполнил: Ашихмин Кирилл, группа P3124

## Цель работы

Освоение методов скрапинга данных из веб-страниц.

## Теоретические сведения

Скрапинг (веб-скрейпинг) — технология получения веб-данных путём
извлечения их со страниц веб-ресурсов.

Объект для скрапинга: <https://news.itmo.ru/ru>

Инструменты:

* requests — библиотека для HTTP-запросов, получает HTML-код страницы;
* BeautifulSoup — разбирает HTML в дерево объектов, по которому можно
  искать элементы по тегу, классу и атрибутам.

## Что нужно собрать

Пункт 3 — общий список новостей:

1. Идентификатор новости (целое число из URL)
2. Название новости
3. Дата размещения
4. URL на страницу с новостью

Пункт 4 — для каждой новости из списка:

1. Идентификатор
2. Название
3. Дата размещения
4. Количество просмотров
5. Текст (с учётом вариативности вёрстки)
6. Теги

Пункт 5: сохранить данные пункта 4 в csv внутри папки `news_content`,
рядом с csv общих данных.

## Подготовка

### Про robots.txt

Перед тем как что-то парсить, нужно посмотреть файл `robots.txt` — в нём сайт
указывает, какие разделы обходить нельзя.

In [1]:
import csv
import os
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

DOMAIN = 'https://news.itmo.ru'

# Представляемся: некоторые сайты отклоняют запросы без User-Agent.
HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36')
}

print(requests.get(f'{DOMAIN}/robots.txt', headers=HEADERS, timeout=30).text)

User-agent: *
#Disallow: /
Disallow: /index.php
Disallow: /cms/
Disallow: /go.php
Disallow: /search/
Disallow: /*unpublished
Host: news.ifmo.ru
Sitemap: http://news.ifmo.ru/module/sitemap.php



Важный вывод из `robots.txt`: раздел `/search/` закрыт для обхода
(`Disallow: /search/`).

Поэтому собирать новости через поисковую строку
(`https://news.itmo.ru/ru/search/?search=...`) — не лучшая идея. Вместо этого
используем ленту главных новостей `/ru/main_news/`, которая в `robots.txt` не
запрещена.

### Вспомогательная функция для запросов

Между запросами делаем паузу. Это не формальность: без пауз мы создаём лишнюю
нагрузку на чужой сервер, и нас могут временно заблокировать.

In [2]:
REQUEST_DELAY = 0.7  # секунд между запросами


def get_soup(url):
    """Скачать страницу и разобрать её в объект BeautifulSoup."""
    time.sleep(REQUEST_DELAY)
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()  # выбросит исключение при ошибке HTTP (404, 500)
    return BeautifulSoup(response.text, 'html.parser')

## Пункт 3. Сбор общего списка новостей

Посмотрим, как устроена одна карточка новости в ленте:

```html
<li>
  <div class="thumb">
    <a href="/ru/science/photonics/news/15018/"><img src="..."/></a>
    <span class="rubric">Главное</span>
  </div>
  <h4>
    <a href="/ru/science/photonics/news/15018/">Эра сверхбыстрых вычислений...</a>
  </h4>
  <time datetime="2026-09-15T16:40:38">15 Сентября 2026</time>
</li>
```

Отсюда берём всё, что нужно для пункта 3: ссылку, заголовок и дату.
Идентификатор — число в конце URL, достаём его регулярным выражением.

Ссылка на новость встречается в `<li>` дважды: в
картинке `div.thumb` и в заголовке `h4`. Берём именно из `h4`, потому что там
есть текст заголовка.

In [3]:
# Шаблон для извлечения id из адреса новости:
# слеш, цифры, конец строки (возможно, с завершающим слешем).
NEWS_URL_PATTERN = re.compile(r'/(\d+)/?$')

# Шаблон для отбора ссылок в ленте. Здесь требование строже: в адресе должен
# быть сегмент /news/. Дело в том, что в ленту попадают не только новости, но
# и анонсы мероприятий (/ru/announce/124118/). У анонсов другая вёрстка: нет
# счётчика просмотров и нет блока с текстом, поэтому они попадали бы в
# результат с пустыми полями. Отбираем только новости.
NEWS_LINK_PATTERN = re.compile(r'/news/(\d+)/?$')


def parse_news_list_page(page_number):
    """Разобрать одну страницу ленты новостей.

    Возвращает список словарей и ссылку на следующую страницу
    (или None, если текущая страница последняя).
    """
    soup = get_soup(f'{DOMAIN}/ru/main_news/{page_number}/')
    news_on_page = []

    for heading in soup.find_all('h4'):
        link = heading.find('a', href=True)
        if link is None:
            continue

        match = NEWS_LINK_PATTERN.search(link['href'])
        if match is None:
            continue  # это не новость (анонс, рубрика или служебная ссылка)

        # Дата лежит в теге <time> рядом с заголовком, внутри того же <li>.
        item = heading.find_parent('li')
        time_tag = item.find('time') if item else None

        news_on_page.append({
            'id': int(match.group(1)),
            'title': link.get_text(strip=True),
            'date': time_tag['datetime'] if time_tag else '',
            'url': DOMAIN + link['href'],
        })

    # Кнопка "Следующая" — вторая ссылка в блоке пагинации.
    # Если её href равен '#', значит страница последняя.
    next_link = soup.find('div', {'class': 'pagination'}).find_all('li')[1].find('a')
    next_href = next_link['href'] if next_link else '#'

    return news_on_page, (None if next_href == '#' else next_href)

In [4]:
# Проверим функцию на одной странице.
first_page, next_page = parse_news_list_page(1)

print('Новостей на странице:', len(first_page))
print('Следующая страница:', next_page)
first_page[0]

Новостей на странице: 9
Следующая страница: /ru/main_news/2/


{'id': 15018,
 'title': 'Эра сверхбыстрых вычислений: в ИТМО появился Институт фотоники',
 'date': '2026-09-15T16:40:38',
 'url': 'https://news.itmo.ru/ru/science/photonics/news/15018/'}

Всё работает. Теперь обойдём несколько страниц ленты.

`MAX_PAGES` ограничивает глубину обхода. Это осознанное решение: полный обход
ленты — это сотни страниц и тысячи запросов к чужому серверу. Для учебной
задачи достаточно нескольких страниц; при необходимости число легко увеличить.

In [5]:
MAX_PAGES = 10

all_news = []

for page_number in range(1, MAX_PAGES + 1):
    news_on_page, next_page = parse_news_list_page(page_number)
    all_news.extend(news_on_page)
    print(f'Страница {page_number}: собрано {len(news_on_page)} новостей '
          f'(всего {len(all_news)})')

    if next_page is None:
        print('Достигнута последняя страница ленты.')
        break

print(f'\nИтого собрано новостей: {len(all_news)}')

Страница 1: собрано 9 новостей (всего 9)


Страница 2: собрано 9 новостей (всего 18)


Страница 3: собрано 9 новостей (всего 27)


Страница 4: собрано 9 новостей (всего 36)


Страница 5: собрано 9 новостей (всего 45)


Страница 6: собрано 9 новостей (всего 54)


Страница 7: собрано 9 новостей (всего 63)


Страница 8: собрано 9 новостей (всего 72)


Страница 9: собрано 9 новостей (всего 81)


Страница 10: собрано 9 новостей (всего 90)

Итого собрано новостей: 90


In [6]:
news_list = pd.DataFrame(all_news)

# Одна и та же новость может попасть в ленту дважды — убираем дубликаты по id.
news_list = news_list.drop_duplicates(subset='id').reset_index(drop=True)

print('Уникальных новостей:', len(news_list))
news_list.head(10)

Уникальных новостей: 90


,id,title,date,url
0,15018,Эра сверхбыстрых вычислений: в ИТМО появился И...,2026-09-15T16:40:38,https://news.itmo.ru/ru/science/photonics/news...
1,15003,Олимпиадный год: итоги самой масштабной приемн...,2026-09-01T19:18:10,https://news.itmo.ru/ru/education/official/new...
2,15001,ITMO CONF 2026: как идеи проходят путь от лабо...,2026-08-31T18:44:11,https://news.itmo.ru/ru/education/trend/news/1...
3,14985,Всем по киберпышке! ИТМО и Яндекс Образование ...,2026-08-18T15:57:07,https://news.itmo.ru/ru/education/cooperation/...
4,14984,Российские школьники взяли четыре медали на Ме...,2026-08-17T16:39:54,https://news.itmo.ru/ru/university_live/achiev...
5,14973,В ИТМО завершили прием на бюджет бакалавриата:...,2026-08-07T14:20:30,https://news.itmo.ru/ru/education/trend/news/1...
6,14967,Студенты ИТМО заняли пять призовых мест на меж...,2026-08-05T16:14:12,https://news.itmo.ru/ru/university_live/achiev...
7,14944,"За дипломом, на сапы и в научный бар. Каким бы...",2026-07-20T16:25:07,https://news.itmo.ru/ru/university_live/leisur...
8,14916,ИТМО получит 196 миллионов рублей на развитие ...,2026-06-30T16:52:15,https://news.itmo.ru/ru/education/official/new...
9,14901,"Для тех, кого не заменит ИИ: в ИТМО стартовал ...",2026-06-20T12:16:31,https://news.itmo.ru/ru/education/official/new...


In [7]:
# Сохраняем общие данные (пункт 3).
news_list.to_csv('news_list.csv', index=False, encoding='utf-8')

print('Сохранено в news_list.csv')

Сохранено в news_list.csv


## Пункт 4. Парсинг страницы каждой новости

Здесь начинается самое интересное — вариативность вёрстки. Разберём, где
что лежит и какие есть подводные камни.

### Количество просмотров

Спрятано внутри тега `<time>` в шапке новости:

```html
<time datetime="2026-09-01T19:18:10+03:00">
    1 Сентября 2026, 19:18<span class="timezone">UTC+3</span><span class="icon eye">5659</span>
</time>
```

То есть искать надо `span` с классом `eye`. При этом если взять
`time.get_text()`, в строку попадут и «UTC+3», и число просмотров — дату
придётся брать из атрибута `datetime`, а не из текста.

### Текст новости

Основной блок — `div` с классом `js-mediator-article`. Но встречаются и другие
варианты вёрстки:

* у части новостей вводный абзац оформлен как `p.lead`, у части — отдельным
  блоком `div.lead`, а у некоторых его нет вовсе;
* интервью свёрстаны через `div.post-content`.

Поэтому ищем блок по очереди в нескольких местах и берём первый, который
нашёлся. Это и есть обработка вариативности вёрстки.

Ещё нюанс: внутри статьи есть подписи к фотографиям, скрипты и стили. Их нужно
удалить, иначе они попадут в текст новости.

### Теги

Здесь легко ошибиться. На первый взгляд теги лежат в блоке с классом
`tags`, и напрашивается `soup.find('div', {'class': 'tags'})`. Но это не
`div`, а `ul` — такой поиск всегда вернёт `None` и теги «потеряются».

Правильно искать без указания тега: `soup.find(class_='tags')`.

Есть и запасной вариант: теги дублируются в мета-тегах в шапке страницы:

```html
<meta property="article:tag" content="Фотоника" />
```

Реализуем оба способа: сначала пробуем список `ul.tags`, если пусто — берём из
мета-тегов.

In [8]:
def parse_news_page(url):
    """Разобрать страницу одной новости."""
    soup = get_soup(url)

    # --- идентификатор из URL ---
    match = NEWS_URL_PATTERN.search(url)
    news_id = int(match.group(1)) if match else None

    # --- заголовок ---
    title_tag = soup.find('h1')
    title = title_tag.get_text(strip=True) if title_tag else ''

    # --- дата: берём из атрибута datetime, а не из текста ---
    time_tag = soup.find('time')
    date = time_tag.get('datetime', '') if time_tag else ''

    # --- просмотры: span с классом eye внутри <time> ---
    views = None
    views_tag = soup.find('span', {'class': 'eye'})
    if views_tag:
        digits = re.sub(r'\D', '', views_tag.get_text())
        views = int(digits) if digits else None

    # --- текст: перебираем возможные варианты вёрстки ---
    body = (soup.find('div', {'class': 'js-mediator-article'})
            or soup.find('div', {'class': 'post-content'})
            or soup.find('article'))

    text = ''
    if body:
        # Убираем всё, что не относится к тексту новости.
        for junk in body.find_all(['script', 'style', 'figure', 'figcaption']):
            junk.decompose()
        for junk in body.find_all('div', {'class': ['copyinfo', 'ya-share2']}):
            junk.decompose()

        paragraphs = [p.get_text(' ', strip=True) for p in body.find_all('p')]
        text = ' '.join(p for p in paragraphs if p)

    # --- теги: сначала список ul.tags, затем мета-теги как запасной вариант ---
    tags = []
    tags_block = soup.find(class_='tags')   # без указания тега: это ul, а не div
    if tags_block:
        tags = [a.get_text(strip=True) for a in tags_block.find_all('a')]
    if not tags:
        tags = [m.get('content') for m in soup.find_all('meta', {'property': 'article:tag'})]

    return {
        'id': news_id,
        'title': title,
        'date': date,
        'views': views,
        'text': text,
        'tags': ', '.join(tags),
    }

In [9]:
# Проверим на одной новости.
example = parse_news_page(news_list.loc[0, 'url'])

for key, value in example.items():
    shown = str(value)
    if len(shown) > 200:
        shown = shown[:200] + f'... (всего {len(str(value))} символов)'
    print(f'{key}: {shown}')

id: 15018
title: Эра сверхбыстрых вычислений: в ИТМО появился Институт фотоники
date: 2026-09-15T16:40:38+03:00
views: 302
text: В 2026 году в Университете ИТМО появился Институт фотоники. Это проектная площадка, которая объединяет исследовательские команды разных факультетов. Главная цель Института — создание отечественных реш... (всего 12175 символов)
tags: Фотоника, Главное, Фотонные технологии, НаукаMain, Передовые технологии


Всё поля заполнены. Теперь обойдём все новости из списка.

Оборачиваем вызов в `try / except`: если одна страница окажется недоступна,
цикл не должен обрушиться и потерять уже собранные данные.

In [10]:
news_content = []
failed_urls = []

for number, row in enumerate(news_list.itertuples(), start=1):
    try:
        news_content.append(parse_news_page(row.url))
    except Exception as error:
        failed_urls.append(row.url)
        print(f'Ошибка при обработке {row.url}: {error}')

    if number % 20 == 0:
        print(f'Обработано {number} из {len(news_list)} новостей...')

print(f'\nГотово. Успешно: {len(news_content)}, с ошибками: {len(failed_urls)}')

Обработано 20 из 90 новостей...


Обработано 40 из 90 новостей...


Обработано 60 из 90 новостей...


Обработано 80 из 90 новостей...



Готово. Успешно: 90, с ошибками: 0


In [11]:
news_content_df = pd.DataFrame(news_content)

print('Размер таблицы:', news_content_df.shape)
news_content_df.head()

Размер таблицы: (90, 6)


,id,title,date,views,text,tags
0,15018,Эра сверхбыстрых вычислений: в ИТМО появился И...,2026-09-15T16:40:38+03:00,303,В 2026 году в Университете ИТМО появился Инсти...,"Фотоника, Главное, Фотонные технологии, НаукаM..."
1,15003,Олимпиадный год: итоги самой масштабной приемн...,2026-09-01T19:18:10+03:00,5708,В ИТМО опубликованы все приказы о зачислении в...,"Магистратура, Бакалавриат, Аспирантура, Главно..."
2,15001,ITMO CONF 2026: как идеи проходят путь от лабо...,2026-08-31T18:44:11+03:00,4003,Трансформация научной идеи в работающий продук...,"Главное, Будущее образования, ГлавноеBottom, И..."
3,14985,Всем по киберпышке! ИТМО и Яндекс Образование ...,2026-08-18T15:57:07+03:00,10703,Университет ИТМО и Яндекс Образование отправля...,"Главное, Геймификация, Фестивали, Яндекс Образ..."
4,14984,Российские школьники взяли четыре медали на Ме...,2026-08-17T16:39:54+03:00,10187,Российские школьники завоевали две золотые и д...,"IOI, Главное, Информационные технологии, Между..."


## Пункт 5. Сохранение в папку news_content

In [12]:
OUTPUT_FOLDER = 'news_content'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

output_path = os.path.join(OUTPUT_FOLDER, 'news_content.csv')

# QUOTE_ALL берёт каждое поле в кавычки: в текстах новостей есть запятые
# и переводы строк, без кавычек csv-файл "поедет".
news_content_df.to_csv(output_path, index=False, encoding='utf-8',
                       quoting=csv.QUOTE_ALL)

print(f'Сохранено: {output_path}')
print(f'Размер файла: {os.path.getsize(output_path) / 1024:.1f} КБ')

print('\nИтоговая структура каталога:')
print('  news_list.csv              — общие данные (пункт 3)')
print(f'  {OUTPUT_FOLDER}/news_content.csv  — подробные данные (пункт 4)')

Сохранено: news_content/news_content.csv
Размер файла: 1503.7 КБ

Итоговая структура каталога:
  news_list.csv              — общие данные (пункт 3)
  news_content/news_content.csv  — подробные данные (пункт 4)


In [13]:
# Проверим, что файл читается обратно без потерь.
check = pd.read_csv(output_path)

print('Прочитано строк:', len(check))
print('Столбцы:', list(check.columns))
check[['id', 'title', 'views']].head()

Прочитано строк: 90
Столбцы: ['id', 'title', 'date', 'views', 'text', 'tags']


,id,title,views
0,15018,Эра сверхбыстрых вычислений: в ИТМО появился И...,303
1,15003,Олимпиадный год: итоги самой масштабной приемн...,5708
2,15001,ITMO CONF 2026: как идеи проходят путь от лабо...,4003
3,14985,Всем по киберпышке! ИТМО и Яндекс Образование ...,10703
4,14984,Российские школьники взяли четыре медали на Ме...,10187


## Анализ собранных данных

In [14]:
print('Всего новостей:', len(news_content_df))
print('Заполненность полей:')
print((news_content_df.notna().sum() / len(news_content_df) * 100).round(1).to_string())

print('\nДлина текста новости (символов):')
print(news_content_df['text'].str.len().describe().round(0).to_string())

print('\nПросмотры:')
print(news_content_df['views'].describe().round(0).to_string())

Всего новостей: 90
Заполненность полей:
id       100.0
title    100.0
date     100.0
views    100.0
text     100.0
tags     100.0

Длина текста новости (символов):
count       90.0
mean      9163.0
std       8762.0
min       1010.0
25%       3400.0
50%       5303.0
75%      12072.0
max      41392.0

Просмотры:
count        90.0
mean      28304.0
std       24241.0
min         303.0
25%       15292.0
50%       21262.0
75%       32820.0
max      150053.0


In [15]:
# Топ-10 новостей по просмотрам.
top_news = news_content_df.nlargest(10, 'views')[['id', 'title', 'views']]
top_news

,id,title,views
12,13310,Как получить образовательный кредит с господде...,150053
16,14779,Миллион уже на первом курсе: ИТМО повысил стип...,123397
19,14692,Новый порядок приема в вузы: что изменится и к...,113003
15,14258,"Яндекс, Сбер, VK и не только: магистратуры ИТМ...",75828
62,14104,"Топ-коллабы, мегапрепод и джун года: кто стал ...",74134
32,14481,Рекорды и ключевые цифры: главные итоги приемн...,70047
22,14601,ИТМО получит 400 млн рублей в рамках программы...,60812
25,14532,В Технологическую долину на территории ИТМО Ха...,57027
44,14366,Все складывается в ИТМО: в Первом неклассическ...,53501
36,14442,В ИТМО подвели первые итоги приемной кампании ...,50647


In [16]:
# Самые частые теги.
all_tags = news_content_df['tags'].str.split(', ').explode()
all_tags = all_tags[all_tags.str.len() > 0]

print('Всего упоминаний тегов:', len(all_tags))
print('Уникальных тегов:', all_tags.nunique())
print('\nТоп-15 тегов:')
print(all_tags.value_counts().head(15).to_string())

Всего упоминаний тегов: 552
Уникальных тегов: 201

Топ-15 тегов:
tags
Главное                      90
ГлавноеBottom                60
Поступление                  18
Искусственный интеллект      18
Абитуриенты                  15
Бакалавриат                  12
Приемная кампания            12
ГлавноеMain1                 12
Рейтинги                     10
Магистратура                  9
ГлавноеTop                    7
Образовательные программы     6
ГлавноеMain2                  6
Образование                   6
УниверситетMain               5


## Выводы

1. Перед скрапингом стоит открыть `robots.txt`. У news.itmo.ru закрыт раздел
`/search/`, то есть именно тот, через который проще всего искать новости по
запросу. Пришлось строить сбор на ленте `/ru/main_news/`, она не запрещена.

2. Идентификатор новости достаётся из URL регулярным выражением
`/(\d+)/?$`: слеш, группа цифр, конец строки (возможно, с завершающим слешем).
Привязка к концу строки нужна, чтобы не поймать число из другой части адреса.

3. Главная сложность задания оказалась в вариативности вёрстки. Три случая,
которые встретились:

* Теги лежат в `ul.tags`. Естественная догадка
  `soup.find('div', {'class': 'tags'})` молча возвращает `None`, теги теряются,
  ошибки при этом никакой. Искать нужно по классу, не указывая тег:
  `soup.find(class_='tags')`. На всякий случай сделан запасной путь через
  мета-теги `article:tag`.
* Блок с текстом называется по-разному. У обычных новостей это
  `div.js-mediator-article`, у интервью `div.post-content`. Перебираем варианты
  по очереди и берём первый найденный.
* Вводный абзац где-то оформлен как `p.lead`, где-то отдельным блоком, где-то
  его нет вовсе. Это разрешилось само: мы собираем все `<p>` внутри блока
  статьи.

4. В ленте лежат не только новости. Первый вариант парсера отбирал ссылки по
шаблону `/(\d+)/?$`, то есть «любой адрес, заканчивающийся числом». Под него
попали анонсы мероприятий вида `/ru/announce/124118/`. Ошибки не возникло,
страницы открывались, но у анонсов другая вёрстка: ни счётчика просмотров, ни
блока со статьёй. В таблице появились строки с пустым текстом и пропусками в
просмотрах, около 5% собранного.

Нашлось это только проверкой заполненности полей после сбора. Шаблон пришлось
ужесточить до `/news/(\d+)/?$`, потребовав сегмент `/news/` в адресе. Вывод на
будущее: полноту собранных данных надо проверять отдельно. Молчаливые пропуски
опаснее явных ошибок, потому что программа при них отрабатывает «успешно».

5. Дату лучше брать из атрибута. В теге `<time>` кроме даты лежат часовой пояс
и счётчик просмотров: `1 Сентября 2026, 19:18UTC+35659`. В атрибуте `datetime`
та же дата в формате ISO 8601, с которым сразу работает `pandas`.

6. Счётчик просмотров спрятан внутри `<time>` в элементе `span.icon.eye`.
Место неочевидное, нашёл просмотром исходного кода страницы.

7. Перед записью текста в csv нужно вычищать разметку: внутри блока статьи
попадаются подписи к фотографиям, кнопки «поделиться» и скрипты. `decompose()`
удаляет элемент вместе с содержимым.

8. `quoting=csv.QUOTE_ALL` при записи текстов обязателен. В новостях есть
запятые, кавычки и переводы строк, без принудительного экранирования файл при
обратном чтении съезжает по столбцам.

9. Скрапинг стоит делать вежливым. Между запросами пауза 0.7 секунды,
передаётся `User-Agent`, глубина обхода ограничена `MAX_PAGES`. Обход всей
ленты означал бы тысячи запросов к чужому серверу и почти наверняка
блокировку.

10. Ошибка на одной странице не должна ронять весь сбор. Каждый запрос обёрнут
в `try / except`, проблемные адреса складываются в отдельный список. Иначе одна
недоступная страница на сотой минуте работы уничтожила бы весь результат.
